In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv(r"C:\Users\anant\OneDrive\Desktop\train_u6lujuX_CVtuZ9i (1).csv")
df

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,LP002978,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural,Y
610,LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural,Y
611,LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban,Y
612,LP002984,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban,Y


In [3]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])   # categorical
    else:
        df[col] = df[col].fillna(df[col].median())    # numerical


In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.select_dtypes(include='object'):
    df[col] = le.fit_transform(df[col])

In [5]:
from imblearn.over_sampling import SMOTE


In [6]:
from sklearn.model_selection import train_test_split

X = df[['Married', 'Education', 'Self_Employed','Credit_History']]
y = df['Loan_Status']



In [7]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X,y)


In [8]:
# Split into train-test sets (80% train, 20% test)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_train_res,y_train_res,test_size=0.2, random_state=42)


In [9]:
len(X_train_res)

844

In [10]:
len(y_train_res)

844

In [11]:
#Applying SVM
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

In [12]:
#Create SVM with Polynomial Kernel
svc_poly = SVC(
    kernel='poly',
    degree=3,          # degree of the polynomial (you can change)
    C=1.0,             # regularization
    gamma='scale'      # kernel coefficient
)

In [13]:
# Fit the model
svc_poly.fit(X_train, y_train)


SVC(kernel='poly')

In [14]:
# Predict
y_pred = svc_poly.predict(X_test)


In [15]:
# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.6627218934911243

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.41      0.58        94
           1       0.57      0.97      0.72        75

    accuracy                           0.66       169
   macro avg       0.76      0.69      0.65       169
weighted avg       0.78      0.66      0.64       169



In [16]:
# Training accuracy
train_pred = svc_poly.predict(X_train)
train_acc = accuracy_score(y_train, train_pred)

# Testing accuracy
test_pred = svc_poly.predict(X_test)
test_acc = accuracy_score(y_test, test_pred)

print("Training Accuracy:", train_acc)
print("Testing Accuracy:", test_acc)

Training Accuracy: 0.7244444444444444
Testing Accuracy: 0.6627218934911243


In [17]:
from sklearn.model_selection import GridSearchCV

In [18]:
# Parameter grid
param_grid = {
    'C': [0.1, 1, 10, 50, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto']
}

# Grid Search
grid = GridSearchCV(svc_poly, param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

# Test prediction
y_pred = grid.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))

Best Parameters: {'C': 0.1, 'gamma': 'scale', 'kernel': 'poly'}
Best Score: 0.7244444444444443
Test Accuracy: 0.6627218934911243


In [19]:
from sklearn.neighbors import KNeighborsClassifier
# KNN model
knn = KNeighborsClassifier()

# Parameter grid
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# GridSearchCV
grid = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

# Predict
y_pred = grid.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))


Best Parameters: {'metric': 'euclidean', 'n_neighbors': 11, 'weights': 'uniform'}
Best Score: 0.7051851851851851
Test Accuracy: 0.6449704142011834


In [20]:
from sklearn.ensemble import RandomForestClassifier
# Random Forest model
rf = RandomForestClassifier(random_state=42)

# Parameter grid
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "bootstrap": [True, False]
}

# GridSearchCV
grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# Fit the model
grid.fit(X_train, y_train)

# Best parameters
print("Best Parameters:", grid.best_params_)

# Best score (cross-validation accuracy)
print("Best CV Accuracy:", grid.best_score_)

# Evaluate on test data
y_pred = grid.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))


Best Parameters: {'bootstrap': True, 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 100}
Best CV Accuracy: 0.7096296296296296
Test Accuracy: 0.6745562130177515


In [21]:
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1

outliers = ((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR)))

outlier_count = outliers.sum().sum()

print("Total outliers:", outlier_count)


Total outliers: 665


In [22]:
len(df)

614

In [23]:
outliers

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,True,False,False,False,False,False,False,False
3,False,False,False,False,True,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,False,True,False,False,False,False,False,False,False,False,False,False,False
610,False,False,False,True,False,False,False,False,False,True,False,False,False
611,False,False,False,False,False,False,False,False,False,False,False,False,False
612,False,False,False,False,False,False,False,False,False,False,False,False,False
